# MNIST 손글씨 분류기 + QMNIST 비교 \😎/

세 모델을 같은 기준으로 학습·평가하고, Streamlit에서 골라 씀.

| 모델 | 내용 |
|---|---|
| **모델 1** | 기존 모델 (baseline) |
| **모델 2** | 모델 1에서 이어서 MNIST로 추가 학습 |
| **모델 3** | QMNIST 학습셋으로 처음부터 학습 |

**실행 순서**: Step 0 → 1 → 2 → 3 → 4 → 5 → 6 (터미널에서 Streamlit 실행)

> 시작 전에 기존 가중치가 `models/mnist_state_dict.pth`에 있어야 함.

## Step 0. 폴더 준비

In [3]:
# 앱에 필요한 폴더 만들기 (이미 있으면 그대로 둠)
import os

APP_DIR = "/Users/sooah/mnist_app"
for sub in ["models", "data", ".streamlit"]:          # 가중치·정보 / 데이터셋 / 테마 설정
    os.makedirs(os.path.join(APP_DIR, sub), exist_ok=True)

# 기존 가중치가 있는지 확인 (모델 1과 모델 2의 출발점)
base_w = os.path.join(APP_DIR, "models", "mnist_state_dict.pth")
print("기존 가중치:", "있음" if os.path.exists(base_w) else "없음 → models 폴더에 넣어 주세요", base_w)

기존 가중치: 있음 /Users/sooah/mnist_app/models/mnist_state_dict.pth


In [4]:
# (선택) 예전 이름(model0~2)으로 만든 결과 파일 정리
# 앱은 models 폴더의 .json을 모두 목록에 올리므로, 예전 파일이 남아 있으면 모델이 중복으로 보입니다.
# 기존 가중치(mnist_state_dict.pth)는 지우지 않습니다.
OLD_FILES = ["model0_baseline.json",
             "model1_mnist_more.json", "model1_mnist_more.pth",
             "model2_qmnist.json", "model2_qmnist.pth"]
for name in OLD_FILES:
    path = os.path.join(APP_DIR, "models", name)
    if os.path.exists(path):
        os.remove(path)
        print("삭제:", name)
print("정리 완료")

삭제: model0_baseline.json
삭제: model1_mnist_more.json
삭제: model1_mnist_more.pth
삭제: model2_qmnist.json
삭제: model2_qmnist.pth
정리 완료


## Step 1. 테마 설정 

In [5]:
%%writefile /Users/sooah/mnist_app/.streamlit/config.toml
# Streamlit 테마 설정: 연한 초록색 화면
[theme]
base = "light"                          # 밝은 테마 기준
primaryColor = "#2E8B57"                # 버튼, 선택 상자 등 강조색 (진한 초록)
backgroundColor = "#EAF7EC"             # 전체 배경 (연한 초록)
secondaryBackgroundColor = "#D3EFD8"    # 입력칸, 표, 펼침 상자 배경 (조금 더 진한 연초록)
textColor = "#1F3A2B"                   # 글자색 (짙은 녹색 계열)

Overwriting /Users/sooah/mnist_app/.streamlit/config.toml


## Step 2. 공통 모듈 `mnist_common.py`
모델 구조, 전처리, 평가 함수

In [6]:
%%writefile /Users/sooah/mnist_app/mnist_common.py
# 공통 모듈: 모델 구조, 전처리, 평가 함수 (학습 스크립트와 Streamlit 앱이 함께 사용)
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms

BASE = Path(__file__).resolve().parent          # mnist_app 폴더
MODEL_DIR = BASE / "models"                     # 가중치(.pth)와 정보(.json) 저장 위치
DATA_DIR = BASE / "data"                        # 데이터셋 다운로드 위치
MEAN, STD = (0.1307,), (0.3081,)                # MNIST 정규화 값 (QMNIST도 같은 값 사용)


class SimpleClassifier(nn.Module):
    """CNN 2층 + 완전연결 2층. 기존 가중치를 그대로 불러와야 하므로 구조를 바꾸지 않음"""
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 28x28 → 14x14
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 14x14 → 7x7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


# 평가·추론용 전처리: PIL 이미지 → 텐서 → 정규화
to_tensor = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])


def pick_device():
    """CUDA → Apple GPU(MPS) → CPU 순서로 사용 가능한 장치 선택"""
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


@torch.no_grad()
def evaluate(model, dataset, device, batch_size=1000):
    """정확도, 1↔7 혼동 건수, 10x10 혼동 행렬 계산"""
    model.eval()
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    conf = torch.zeros(10, 10, dtype=torch.long)      # conf[정답, 예측]
    for x, y in loader:
        pred = model(x.to(device)).argmax(dim=1).cpu()
        conf += torch.bincount(y * 10 + pred, minlength=100).view(10, 10)
    total = int(conf.sum())
    return {
        "accuracy": float(conf.trace()) / total,
        "n": total,
        "1->7": int(conf[1, 7]),                      # 정답 1을 7로 틀린 수
        "7->1": int(conf[7, 1]),                      # 정답 7을 1로 틀린 수
        "confusion": conf.tolist(),
    }

Overwriting /Users/sooah/mnist_app/mnist_common.py


## Step 3. 학습 스크립트 `train_models.py`
세 모델을 만들고, **같은 평가셋**(MNIST 테스트 1만 + QMNIST 새 테스트 5만)으로 평가해 `models/*.json`에 저장. 앱은 이 JSON 파일로 모델 목록을 만듦.

In [7]:
%%writefile /Users/sooah/mnist_app/train_models.py
# 학습 스크립트: 모델 1(기존), 모델 2(모델 1 + 추가 학습), 모델 3(QMNIST)을 만들고 같은 기준으로 평가
# 실행: mnist_app 폴더에서  python train_models.py
import json
import random
import time

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from mnist_common import (DATA_DIR, MEAN, MODEL_DIR, STD, SimpleClassifier,
                          evaluate, pick_device, to_tensor)

# ---------------- 실험 설정 (결과 JSON에 그대로 기록됨) ----------------
SEED = 42
BASELINE_WEIGHTS = "mnist_state_dict.pth"   # 기존 가중치 (models 폴더 안)

M2_EXTRA_EPOCHS = 5      # 모델 2: 모델 1에 추가로 학습할 에폭 수
M2_LR = 1e-4             # 모델 2: 이미 학습된 모델이라 작은 학습률로 이어서 학습
M2_AUGMENT = False       # True: 회전·이동·크기·기울임 증강을 넣어 추가 학습 (1·7 개선용)

M3_EPOCHS = 5            # 모델 3: QMNIST 학습셋으로 처음부터 학습할 에폭 수
M3_LR = 1e-3
BATCH_SIZE = 128
# ---------------------------------------------------------------------

# 재현성을 위해 난수 고정
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = pick_device()
print("장치:", device)

# 증강 전처리 (M2_AUGMENT=True일 때만 사용)
augment = transforms.Compose([
    transforms.RandomAffine(degrees=10, translate=(0.1, 0.1), scale=(0.9, 1.1), shear=10),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# 모든 모델을 같은 평가셋으로 비교 (통제 변수)
EVAL_SETS = {
    "MNIST 테스트 (1만)": datasets.MNIST(DATA_DIR, train=False, download=True, transform=to_tensor),
    "QMNIST 새 테스트 (5만)": datasets.QMNIST(DATA_DIR, what="test50k", download=True, transform=to_tensor),
}


def train(model, dataset, epochs, lr):
    """분류 학습 루프 (Adam + 교차 엔트로피)"""
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    for ep in range(epochs):
        model.train()
        total, n, t0 = 0.0, 0, time.time()
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = loss_fn(model(x), y)
            loss.backward()
            opt.step()
            total += loss.item() * len(y)
            n += len(y)
        print(f"  epoch {ep + 1}/{epochs}  loss {total / n:.4f}  ({time.time() - t0:.0f}초)")
    model.eval()
    return model


def save(model, key, name, description, train_config, weights_file=None):
    """가중치 저장 + 평가 결과와 실험 조건을 JSON으로 저장 (앱이 이 JSON으로 목록을 만듦)"""
    MODEL_DIR.mkdir(exist_ok=True)
    if weights_file is None:                       # 새로 학습한 모델만 가중치를 새로 저장
        weights_file = f"{key}.pth"
        torch.save(model.state_dict(), MODEL_DIR / weights_file)
    metrics = {k: evaluate(model, ds, device) for k, ds in EVAL_SETS.items()}
    meta = {
        "key": key, "name": name, "description": description,
        "weights": weights_file, "train_config": train_config, "metrics": metrics,
        "conditions": {"torch": torch.__version__, "device": str(device), "seed": SEED},
    }
    with open(MODEL_DIR / f"{key}.json", "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)
    print(f"[{name}] 저장")
    for k, m in metrics.items():
        print(f"  {k}: 정확도 {m['accuracy']:.2%}  1→7 {m['1->7']}건  7→1 {m['7->1']}건")


def load_baseline():
    """기존 가중치를 불러옴"""
    model = SimpleClassifier().to(device)
    state = torch.load(MODEL_DIR / BASELINE_WEIGHTS, map_location=device, weights_only=True)
    model.load_state_dict(state)
    return model.eval()


if __name__ == "__main__":
    # 모델 1: 기존 모델을 같은 평가셋으로 측정 (비교 기준선, 가중치는 그대로 사용)
    print("\n=== 모델 1: 기존 모델 (baseline) 평가 ===")
    save(load_baseline(), "model1_baseline", "모델 1: 기존 모델 (baseline)",
         "처음 학습한 MNIST 모델. 다른 모델의 비교 기준", {"source": BASELINE_WEIGHTS},
         weights_file=BASELINE_WEIGHTS)

    # 모델 2: 모델 1에서 이어서 MNIST 학습셋으로 추가 학습
    print(f"\n=== 모델 2: 모델 1 + {M2_EXTRA_EPOCHS}에폭 추가 학습 (증강 {M2_AUGMENT}) ===")
    m2_data = datasets.MNIST(DATA_DIR, train=True, download=True,
                             transform=augment if M2_AUGMENT else to_tensor)
    m2 = train(load_baseline(), m2_data, M2_EXTRA_EPOCHS, M2_LR)
    save(m2, "model2_mnist_more", "모델 2: 모델 1 + 추가 학습",
         f"모델 1 가중치에서 MNIST로 {M2_EXTRA_EPOCHS}에폭 더 학습 (증강 {'사용' if M2_AUGMENT else '없음'})",
         {"init": BASELINE_WEIGHTS, "data": "MNIST train", "epochs": M2_EXTRA_EPOCHS,
          "lr": M2_LR, "augment": M2_AUGMENT, "batch_size": BATCH_SIZE})

    # 모델 3: QMNIST 학습셋으로 처음부터 학습
    print(f"\n=== 모델 3: QMNIST로 처음부터 {M3_EPOCHS}에폭 학습 ===")
    torch.manual_seed(SEED)                         # 모델 3의 초기 가중치도 고정
    m3_data = datasets.QMNIST(DATA_DIR, what="train", download=True, transform=to_tensor)
    m3 = train(SimpleClassifier().to(device), m3_data, M3_EPOCHS, M3_LR)
    save(m3, "model3_qmnist", "모델 3: QMNIST 학습",
         f"QMNIST 학습셋(6만)으로 처음부터 {M3_EPOCHS}에폭 학습",
         {"init": "random", "data": "QMNIST train", "epochs": M3_EPOCHS,
          "lr": M3_LR, "augment": False, "batch_size": BATCH_SIZE})

Overwriting /Users/sooah/mnist_app/train_models.py


## Step 4. 학습 실행

In [8]:
# 앱 폴더로 이동한 뒤 학습 스크립트 실행 (모델 1·2·3의 .json과 모델 2·3의 .pth 생성)
%cd /Users/sooah/mnist_app
import sys
!{sys.executable} train_models.py

/Users/sooah/mnist_app
장치: mps

=== 모델 1: 기존 모델 (baseline) 평가 ===
[모델 1: 기존 모델 (baseline)] 저장
  MNIST 테스트 (1만): 정확도 99.33%  1→7 2건  7→1 2건
  QMNIST 새 테스트 (5만): 정확도 98.98%  1→7 5건  7→1 8건

=== 모델 2: 모델 1 + 5에폭 추가 학습 (증강 False) ===
  epoch 1/5  loss 0.0256  (5초)
  epoch 2/5  loss 0.0229  (5초)
  epoch 3/5  loss 0.0186  (5초)
  epoch 4/5  loss 0.0184  (5초)
  epoch 5/5  loss 0.0171  (5초)
[모델 2: 모델 1 + 추가 학습] 저장
  MNIST 테스트 (1만): 정확도 99.38%  1→7 0건  7→1 2건
  QMNIST 새 테스트 (5만): 정확도 99.14%  1→7 3건  7→1 8건

=== 모델 3: QMNIST로 처음부터 5에폭 학습 ===
  epoch 1/5  loss 0.2563  (5초)
  epoch 2/5  loss 0.0846  (5초)
  epoch 3/5  loss 0.0632  (5초)
  epoch 4/5  loss 0.0513  (5초)
  epoch 5/5  loss 0.0422  (5초)
[모델 3: QMNIST 학습] 저장
  MNIST 테스트 (1만): 정확도 99.12%  1→7 1건  7→1 2건
  QMNIST 새 테스트 (5만): 정확도 98.92%  1→7 7건  7→1 4건


## Step 5. Streamlit 앱 `mnist_ui.py`
모델 선택, 모델별 평가 비교, 4가지 입력 방식(그리기 / MNIST 샘플 / QMNIST 새 샘플 / 업로드), 모든 모델 예측 비교를 제공.

In [9]:
%%writefile /Users/sooah/mnist_app/mnist_ui.py
# Streamlit 앱: 학습된 모델 목록에서 하나를 골라 손글씨 숫자를 분류
# 실행: mnist_app 폴더에서  streamlit run mnist_ui.py
import json

import numpy as np
import pandas as pd
import streamlit as st
import torch
from PIL import Image, ImageOps
from streamlit_drawable_canvas import st_canvas
from torchvision import datasets

from mnist_common import DATA_DIR, MODEL_DIR, SimpleClassifier, to_tensor

# ---------------- 페이지 설정과 테마 (반드시 첫 Streamlit 명령) ----------------
st.set_page_config(page_title="MNIST 손글씨 분류기", page_icon="✏️", layout="centered")

# config.toml이 적용되지 않아도 처음부터 연한 초록색으로 보이도록 CSS로 한 번 더 지정
st.markdown(
    """
    <style>
    .stApp { background-color: #EAF7EC; }                       /* 전체 배경: 연한 초록 */
    [data-testid="stHeader"] { background-color: #EAF7EC; }     /* 상단 바도 같은 색 */
    [data-testid="stExpander"] details { background-color: #D3EFD8; border-radius: 8px; }
    </style>
    """,
    unsafe_allow_html=True,
)


# ---------------- 함수 ----------------
def to_mnist_style(img):
    """흰 글씨/검은 배경 이미지를 MNIST처럼 글씨만 잘라 20x20으로 줄이고 28x28 가운데에 놓는다"""
    arr = np.array(img)
    ys, xs = np.nonzero(arr > 30)
    if len(xs) == 0:
        return None                                   # 아무것도 안 그림
    digit = img.crop((xs.min(), ys.min(), xs.max() + 1, ys.max() + 1))
    scale = 20 / max(digit.size)
    digit = digit.resize((max(1, round(digit.width * scale)), max(1, round(digit.height * scale))), Image.LANCZOS)
    canvas = Image.new("L", (28, 28), 0)
    canvas.paste(digit, ((28 - digit.width) // 2, (28 - digit.height) // 2))
    return canvas


def list_models():
    """models 폴더의 *.json(학습 스크립트가 만든 정보 파일)을 읽어 모델 목록 생성"""
    metas = []
    for path in sorted(MODEL_DIR.glob("*.json")):     # model1_, model2_, model3_ 순서로 정렬됨
        with open(path, encoding="utf-8") as f:
            metas.append(json.load(f))
    return metas


@st.cache_resource
def load_model(weights_file):
    """가중치 파일별로 한 번만 불러와 캐시"""
    model = SimpleClassifier()
    model.load_state_dict(torch.load(MODEL_DIR / weights_file, map_location="cpu", weights_only=True))
    return model.eval()


@st.cache_resource
def load_mnist_test():
    return datasets.MNIST(DATA_DIR, train=False, download=True)


@st.cache_resource
def load_qmnist_test50k():
    return datasets.QMNIST(DATA_DIR, what="test50k", download=True)


def predict(model, image):
    """28x28 PIL 이미지 → 클래스별 확률(numpy)"""
    with torch.no_grad():
        return torch.softmax(model(to_tensor(image).unsqueeze(0)), dim=1)[0].numpy()


st.title("MNIST 손글씨 숫자 분류기")

# ---------------- 모델 선택 ----------------
metas = list_models()
if not metas:
    st.error("models 폴더에 모델 정보(.json)가 없습니다. 먼저 `python train_models.py`를 실행하세요.")
    st.stop()

meta = st.selectbox("모델 선택", metas, format_func=lambda m: m["name"])
st.caption(meta["description"])
model = load_model(meta["weights"])

# 모든 모델의 평가 결과를 한 표로 비교 (같은 평가셋, 같은 기준)
with st.expander("모델별 평가 결과 비교"):
    rows = []
    for m in metas:
        row = {"모델": m["name"]}
        for set_name, r in m["metrics"].items():
            row[f"{set_name} 정확도"] = f"{r['accuracy']:.2%}"
            row[f"{set_name} 1→7"] = r["1->7"]
            row[f"{set_name} 7→1"] = r["7->1"]
        rows.append(row)
    st.dataframe(pd.DataFrame(rows).set_index("모델"))

# ---------------- 입력 ----------------
source = st.radio("입력 방식", ["직접 그리기", "MNIST 테스트 샘플", "QMNIST 새 테스트 샘플", "이미지 업로드"],
                  horizontal=True)
image, label = None, None

if source == "직접 그리기":
    st.caption("검은 칸에 숫자 하나를 크게 그려 보세요. 휴지통 아이콘으로 지울 수 있어요.")
    result = st_canvas(                               # 캔버스는 MNIST처럼 검은 배경·흰 글씨 유지
        stroke_width=18, stroke_color="#FFFFFF", background_color="#000000",
        width=280, height=280, drawing_mode="freedraw", key="canvas",
        return_image_data=True,
    )
    if result.image_data is not None:
        drawn = Image.fromarray(result.image_data[:, :, 0].astype(np.uint8))   # 흰 글씨 채널만 사용
        image = to_mnist_style(drawn)

elif source == "MNIST 테스트 샘플":
    idx = st.number_input("샘플 번호 (0~9999)", 0, 9999, 0)
    image, label = load_mnist_test()[idx]

elif source == "QMNIST 새 테스트 샘플":
    idx = st.number_input("샘플 번호 (0~49999)", 0, 49999, 0)
    image, label = load_qmnist_test50k()[idx]

else:
    file = st.file_uploader("숫자 이미지", type=["png", "jpg", "jpeg"])
    if file:
        img = Image.open(file).convert("L")
        if st.checkbox("흰 배경에 검은 글씨면 체크 (색 반전)", value=True):
            img = ImageOps.invert(img)
        image = to_mnist_style(img)

# ---------------- 예측 ----------------
if image is not None:
    col1, col2 = st.columns(2)
    col1.image(image.resize((140, 140), Image.NEAREST), caption="모델이 보는 28x28 이미지")
    probs = predict(model, image)
    pred = int(probs.argmax())
    col2.metric("예측", pred, f"확신도 {probs[pred]:.1%}")
    if label is not None:
        col2.write(f"정답: **{label}** {'✅' if pred == label else '❌'}")
    st.bar_chart({"확률": probs}, x_label="숫자", y_label="확률", color="#2E8B57")

    # 같은 이미지에 대한 모든 모델의 예측을 나란히 비교
    if st.checkbox("모든 모델 예측 비교"):
        compare = []
        for m in metas:
            p = predict(load_model(m["weights"]), image)
            compare.append({"모델": m["name"], "예측": int(p.argmax()),
                            "확신도": f"{p.max():.1%}", "1의 확률": f"{p[1]:.1%}", "7의 확률": f"{p[7]:.1%}"})
        st.dataframe(pd.DataFrame(compare).set_index("모델"))

Overwriting /Users/sooah/mnist_app/mnist_ui.py


## Step 6. streamlit 앱 실행 (터미널)
```bash
cd /Users/sooah/mnist_app
streamlit run mnist_ui.py
```

In [10]:
%%writefile /Users/sooah/mnist_app/train_aug.py
# 증강 학습 스크립트: 모델 2 → 모델 4, 모델 3 → 모델 5 (각각 증강을 넣어 5에폭 추가 학습)
# 모델 1·2·3은 다시 학습하지 않음. train_models.py의 학습·저장·평가 함수와 증강 설정을 그대로 재사용
# 실행: mnist_app 폴더에서  python train_aug.py
import torch
from torchvision import datasets

from mnist_common import DATA_DIR, MODEL_DIR, SimpleClassifier
from train_models import BATCH_SIZE, SEED, augment, device, save, train  # 같은 함수·같은 평가셋 사용

# ---------------- 실험 설정 ----------------
AUG_EPOCHS = 5           # 모델 4·5 모두 5에폭 추가 학습
AUG_LR = 1e-4            # 이미 학습된 모델에서 이어 가므로 작은 학습률
# 증강 내용: 회전 ±10°, 이동 ±10%, 크기 0.9~1.1배, 기울임 ±10° (train_models.py의 augment)
# -------------------------------------------


def load_weights(weights_file):
    """models 폴더의 가중치를 불러옴. 없으면 무엇을 먼저 실행해야 하는지 알려 줌"""
    path = MODEL_DIR / weights_file
    if not path.exists():
        raise FileNotFoundError(f"{path} 가 없습니다. 먼저 train_models.py로 모델 2·3을 만들어 주세요.")
    model = SimpleClassifier().to(device)
    model.load_state_dict(torch.load(path, map_location=device, weights_only=True))
    return model


if __name__ == "__main__":
    # 모델 4: 모델 2에서 이어서, 증강한 MNIST로 5에폭
    print(f"\n=== 모델 4: 모델 2 + 증강 {AUG_EPOCHS}에폭 ===")
    torch.manual_seed(SEED)                         # 섞는 순서와 증강의 무작위성 고정
    m4_data = datasets.MNIST(DATA_DIR, train=True, download=True, transform=augment)
    m4 = train(load_weights("model2_mnist_more.pth"), m4_data, AUG_EPOCHS, AUG_LR)
    save(m4, "model4_mnist_aug", "모델 4: 모델 2 + 증강 학습",
         f"모델 2 가중치에서 증강한 MNIST로 {AUG_EPOCHS}에폭 더 학습",
         {"init": "model2_mnist_more.pth", "data": "MNIST train", "epochs": AUG_EPOCHS,
          "lr": AUG_LR, "augment": True, "batch_size": BATCH_SIZE})

    # 모델 5: 모델 3에서 이어서, 증강한 QMNIST로 5에폭
    print(f"\n=== 모델 5: 모델 3 + 증강 {AUG_EPOCHS}에폭 ===")
    torch.manual_seed(SEED)
    m5_data = datasets.QMNIST(DATA_DIR, what="train", download=True, transform=augment)
    m5 = train(load_weights("model3_qmnist.pth"), m5_data, AUG_EPOCHS, AUG_LR)
    save(m5, "model5_qmnist_aug", "모델 5: 모델 3 + 증강 학습",
         f"모델 3 가중치에서 증강한 QMNIST로 {AUG_EPOCHS}에폭 더 학습",
         {"init": "model3_qmnist.pth", "data": "QMNIST train", "epochs": AUG_EPOCHS,
          "lr": AUG_LR, "augment": True, "batch_size": BATCH_SIZE})

Overwriting /Users/sooah/mnist_app/train_aug.py


In [11]:
%cd /Users/sooah/mnist_app
import sys
!{sys.executable} train_aug.py

/Users/sooah/mnist_app
장치: mps

=== 모델 4: 모델 2 + 증강 5에폭 ===
  epoch 1/5  loss 0.3153  (7초)
  epoch 2/5  loss 0.2163  (7초)
  epoch 3/5  loss 0.1795  (7초)
  epoch 4/5  loss 0.1536  (7초)
  epoch 5/5  loss 0.1391  (7초)
[모델 4: 모델 2 + 증강 학습] 저장
  MNIST 테스트 (1만): 정확도 99.40%  1→7 0건  7→1 2건
  QMNIST 새 테스트 (5만): 정확도 99.13%  1→7 4건  7→1 3건

=== 모델 5: 모델 3 + 증강 5에폭 ===
  epoch 1/5  loss 0.3115  (7초)
  epoch 2/5  loss 0.2115  (7초)
  epoch 3/5  loss 0.1748  (7초)
  epoch 4/5  loss 0.1532  (7초)
  epoch 5/5  loss 0.1371  (7초)
[모델 5: 모델 3 + 증강 학습] 저장
  MNIST 테스트 (1만): 정확도 99.23%  1→7 0건  7→1 3건
  QMNIST 새 테스트 (5만): 정확도 99.12%  1→7 2건  7→1 3건


| 항목 | 모델 1: baseline | 모델 2: 모델 1 + 추가 학습 | 모델 3: QMNIST 학습 | 모델 4: 모델 2 + 증강 | 모델 5: 모델 3 + 증강 |
|---|---|---|---|---|---|
| 학습한 곳 | D46 노트북 5.2절 | `train_models.py` | `train_models.py` | `train_aug.py` | `train_aug.py` |
| 가중치 파일 | `mnist_state_dict.pth` | `model2_mnist_more.pth` | `model3_qmnist.pth` | `model4_mnist_aug.pth` | `model5_qmnist_aug.pth` |
| 시작 가중치 | 무작위 | **모델 1에서 이어서** | 무작위 | **모델 2에서 이어서** | **모델 3에서 이어서** |
| 학습 데이터 | MNIST 학습셋 6만 | MNIST 학습셋 6만 (한 번 더) | **QMNIST 학습셋 6만** | MNIST 학습셋 6만 | QMNIST 학습셋 6만 |
| 에폭 | **3** | +5 (누적 8) | **5** | +5 (누적 13) | +5 (누적 10) |
| 학습률 | **1e-3** | **1e-4** (이어서 학습하므로 작게) | **1e-3** | **1e-4** | **1e-4** |
| 옵티마이저 / 손실 | Adam / 교차 엔트로피 | Adam / 교차 엔트로피 | Adam / 교차 엔트로피 | Adam / 교차 엔트로피 | Adam / 교차 엔트로피 |
| 배치 크기 | **64** | **128** | **128** | **128** | **128** |
| 데이터 증강 | 없음 | 없음 | 없음 | **있음** | **있음** |
| 확인하려는 것 | 비교 기준 | 같은 데이터로 **더 오래** 배우면 나아지나? | **데이터를 바꿔** 처음부터 배우면 달라지나? | 모델 2에 **증강**을 더하면 나아지나? | 모델 3에 **증강**을 더하면 나아지나? |

**증강 내용 (모델 4·5 공통)**: 학습할 때마다 이미지를 회전 ±10°, 이동 ±10%, 크기 0.9~1.1배, 기울임 ±10° 범위에서 무작위로 바꿔서 보여 준다. 평가와 앱에서는 증강 없이 원래 이미지를 쓴다.

**비교 짝**
- 모델 1 → 2: 추가 학습의 효과
- 모델 1 → 3: 데이터(QMNIST) + 학습 조건 차이의 효과 (에폭·배치 크기가 달라 완전히 분리되지는 않음)
- 모델 2 → 4: MNIST에서 증강의 효과 (+5에폭 추가 학습 포함)
- 모델 3 → 5: QMNIST에서 증강의 효과 (+5에폭 추가 학습 포함)